In [1]:
from pathlib import Path
from datetime import date

import pandas as pd
import numpy as np

from maps import zsdkap_new_columns_names, zsdkap_dtypes, vbap_new_columns_names, mb5td_new_columns_names, mb5td_dtypes, zek103_new_columns_names, zek103_dtypes, mb52_dtypes, mb52_new_columns_names
from py_rfc_methods import get_delivery_plants_and_special_stock_indicators_df
from helper_functions import clean_number

In [2]:
SAP_SYSTEM = "P11_SSO"
STORAGE_LOCATIONS = ['0004', '0005']

In [40]:
source_files_dir = Path(r"P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\18_MISSING_STOCK_ORDERS_IDENTIFIER")
output_files_dir = Path(r"P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\18_MISSING_STOCK_ORDERS_IDENTIFIER\output")

# 1. Define filenames in ONE place using a dictionary
source_file_names = {
    "zsdkap": "ZSDKAP.csv",
    "mb5td": "MB5TD.XLSX",
    "zek103": "ZEK103.XLSX",
    "mb52": "MB52.XLSX",
}

output_file_names = {
    "df": "df.xlsx",
}

source_files = {key: source_files_dir / name for key, name in source_file_names.items()}
output_files = {key: output_files_dir / name for key, name in output_file_names.items()}

In [4]:
product_names = ('ZAR')
mrp_controllers = ('L2B', 'L2R', 'LI2', 'LI4', 'LI7')

In [5]:
zsdkap = pd.read_csv(source_files["zsdkap"], dtype=zsdkap_dtypes, sep=';', encoding='MacRoman')

In [6]:
df = zsdkap.copy()

In [7]:
df = df.rename(columns=zsdkap_new_columns_names)
df = df[(df['mrp_controller'].isin(mrp_controllers)) & (df['mat_description'].str.startswith(product_names))].reset_index()

In [8]:
df['orders_quantity'] = df['orders_quantity'].apply(clean_number)
df['orders_quantity'] = df['orders_quantity'].astype(int)
df['customer_order_position'] = df['customer_order_position'].str.zfill(6)
df['dispatch_date'] = pd.to_datetime(df['dispatch_date'], dayfirst=True, errors='coerce')
df['creation_date'] = pd.to_datetime(df['creation_date'], dayfirst=True, errors='coerce')

In [9]:
columns_to_drop = ['Land', 'Postleitzahl', 'Ort', 'Strasse', 'Name', 'UPS', 'Materialgruppe', 'Materialgruppenbezeichnung', 'Bestellnummer', 'Versandbedingung', 'Uhrzeiten', 'Kopfnotiz 3', 'Kopfnotiz 4', 'Gewicht', 'Volumen', 'Lieferpriorität', 'Route', 'Transportzone WE', 'Wskaźnik przetw. specj.', 'ID kontenera', 'Spedition']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [10]:
delivery_plants = get_delivery_plants_and_special_stock_indicators_df(SAP_SYSTEM, df['customer_order_number'].tolist(), 1000, 100)
delivery_plants = delivery_plants.rename(columns=vbap_new_columns_names)

2026-07-09 17:13:35 | INFO     | SAP_CONN | Buduję parametry dla systemu P11_SSO: {'mshost': 'rffsp11s.sap.roto-frank.com', 'msserv': 'sapmsP11', 'sysid': 'P11', 'group': 'ROTO_FRANK', 'client': '151', 'lang': 'PL', 'snc_mode': '1', 'snc_qop': '8', 'snc_partnername': ' ', 'timeout': '300'}
2026-07-09 17:13:35 | INFO     | SAP_CONN | Połączenie z SAP nawiązane (system=P11_SSO).
2026-07-09 17:13:36 | INFO     | SAP_CONN | Połączenie SAP zamknięte (system=P11_SSO).


In [11]:
df = pd.merge(df, delivery_plants, how='left', on=['customer_order_number', 'customer_order_position'])
df.dropna(subset=['delivery_plant'], inplace=True)

In [12]:
df = df[df['special_stock_indicator'] != 'E']

In [13]:
# Newest dispatch_date first, oldest creation_date second
df_sorted = df.sort_values(by=['dispatch_date', 'creation_date'], ascending=[False, True])

In [14]:
df_grouped = df.groupby(['delivery_plant', 'mat_number', 'dispatch_date'], as_index=False).agg({
    'orders_quantity': 'sum',
    'mat_description': 'first',
})

In [15]:
df_grouped.head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description
0,0301,2044034,2026-07-09,1,ZAR M 078/xxx R4R7WL
1,0301,283953,2026-07-10,3,ZAR M 114/xxx 4373
2,0301,283953,2026-07-13,3,ZAR M 114/xxx 4373
3,0301,283953,2026-07-14,2,ZAR M 114/xxx 4373
4,0301,283953,2026-07-16,1,ZAR M 114/xxx 4373


In [16]:
mb5td_df = pd.read_excel(source_files["mb5td"], dtype=mb5td_dtypes)
mb5td_df.rename(columns=mb5td_new_columns_names, inplace=True)

In [17]:
sap_list = df['mat_number'].tolist()
mb5td_df = mb5td_df[mb5td_df['mat_number'].isin(sap_list)]

In [18]:
mb5td_df.head()

,mat_number,Opis materiału,plant,Nazwa 1,purchase_order_number,purchase_order_position,supplying_plant,special_stock_indicator,transit_quantity,Jedn. miary zamów.,Kwota w WKr,Waluta,Ilość zamówienia,Wartość netto zamów.,Waluta.1
118,538221,ZAR M 114/xxx R6R8,1201,ROTO Bauelemente St. Avold FR,4500135419,150,2101,E,2.0,SZT,70.80,EUR,2,70.80,EUR
128,283965,ZAR M 134/xxx 6x84,0301,RotoFrank DST Produktions-GmbH,4500135630,60,2101,NaN,10.0,SZT,357.60,EUR,10,357.60,EUR
184,283964,ZAR M 114/xxx 6x84,1201,ROTO Bauelemente St. Avold FR,4500136510,10,2101,E,2.0,SZT,64.34,EUR,2,64.34,EUR
204,538204,ZAR M 065/xxx R6R8,0301,RotoFrank DST Produktions-GmbH,4500136706,10,2101,NaN,10.0,SZT,265.70,EUR,10,265.70,EUR
207,283958,ZAR M 054/xxx 6x84,0301,RotoFrank DST Produktions-GmbH,4500136710,30,2101,NaN,10.0,SZT,209.80,EUR,10,209.80,EUR


In [19]:
zek103_df = pd.read_excel(source_files['zek103'], dtype=zek103_dtypes)
zek103_df.rename(columns=zek103_new_columns_names, inplace=True)

In [20]:
zek103_df['po_delivery_date'] = pd.to_datetime(zek103_df['po_delivery_date'], dayfirst=True, errors='coerce')
zek103_df['po_dispatch_date'] = pd.to_datetime(zek103_df['po_dispatch_date'], dayfirst=True, errors='coerce')
zek103_df = zek103_df[zek103_df['mat_number'].isin(sap_list)]

In [21]:
cols_to_drop = ['Szuk. ciąg', 'LFT', 'Best-Mg', 'Bestät. Menge', 'ME', 'Best. Liefdat.']
zek103_df = zek103_df.drop(columns=[col for col in cols_to_drop if col in zek103_df.columns])

In [22]:
zek103_df.head(5)

,purchase_order_number,purchase_order_position,mat_number,mat_description_zek103,po_delivery_date,po_quantity,plant,po_dispatch_date,customer_order_number,customer_order_position
97,4500118212,20,538188,ZAR M 074/xxx R4R7,2026-04-07,0.0,0301,2026-04-01,<NA>,0
136,4500135314,30,741467,ZAR M 094/xxx Qx__,2026-07-17,10.0,0301,2026-07-15,<NA>,0
148,4500135630,60,283965,ZAR M 134/xxx 6x84,2026-07-10,10.0,0301,2026-07-08,<NA>,0
154,4500135877,20,741469,ZAR M 114/xxx Qx__,2026-07-15,10.0,0301,2026-07-13,<NA>,0
159,4500135884,30,283953,ZAR M 114/xxx 4373,2026-07-17,20.0,0301,2026-07-15,<NA>,0


In [23]:
# Drop purchase orders linked to special customer requirements
zek103_df = zek103_df[zek103_df['customer_order_number'].isna()]

In [24]:
zek103_df = zek103_df.groupby(['po_delivery_date', 'plant', 'mat_number'], as_index=False).agg({
    'po_quantity': 'sum',
    'po_dispatch_date': 'first',
    'mat_description_zek103': 'first',

})

In [25]:
zsdkap_zek103_merged_df = pd.merge(df_grouped, zek103_df, how='outer', left_on=['delivery_plant', 'mat_number', 'dispatch_date'], right_on=['plant', 'mat_number', 'po_delivery_date'])

In [26]:
zsdkap_zek103_merged_df['delivery_plant'] = zsdkap_zek103_merged_df['delivery_plant'].combine_first(zsdkap_zek103_merged_df['plant'])
zsdkap_zek103_merged_df['dispatch_date'] = zsdkap_zek103_merged_df['dispatch_date'].combine_first(zsdkap_zek103_merged_df['po_delivery_date'])
zsdkap_zek103_merged_df['mat_description'] = zsdkap_zek103_merged_df['mat_description'].combine_first(zsdkap_zek103_merged_df['mat_description_zek103'])

In [27]:
zsdkap_zek103_merged_df['orders_quantity'] = zsdkap_zek103_merged_df['orders_quantity'].fillna(0)
zsdkap_zek103_merged_df['po_quantity'] = zsdkap_zek103_merged_df['po_quantity'].fillna(0)

In [28]:
zsdkap_zek103_merged_df = zsdkap_zek103_merged_df.drop(columns=['plant', 'po_delivery_date', 'mat_description_zek103'])

In [29]:
zsdkap_zek103_merged_df.head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date
0,0301,2044034,2026-07-09,1.0,ZAR M 078/xxx R4R7WL,0.0,NaT
1,0301,283953,2026-07-10,3.0,ZAR M 114/xxx 4373,0.0,NaT
2,0301,283953,2026-07-13,3.0,ZAR M 114/xxx 4373,0.0,NaT
3,0301,283953,2026-07-14,2.0,ZAR M 114/xxx 4373,0.0,NaT
4,0301,283953,2026-07-16,1.0,ZAR M 114/xxx 4373,0.0,NaT


In [30]:
mb52_df = pd.read_excel(source_files['mb52'], dtype=mb52_dtypes)
mb52_df.rename(columns=mb52_new_columns_names, inplace=True)
mb52_df = mb52_df[(mb52_df['mat_number'].isin(sap_list)) & (mb52_df['customer_order_number'].isna()) & (mb52_df['storage_location'].isin(STORAGE_LOCATIONS))]

In [32]:
mb52_df = mb52_df.groupby(['plant', 'mat_number'], as_index=False).agg({
    'stock_quantity': 'sum',
})

In [34]:
mb52_df.head()

,plant,mat_number,stock_quantity
0,0301,2044034,3.0
1,0301,283953,19.0
2,0301,283954,3.0
3,0301,283955,2.0
4,0301,283956,7.0


In [37]:
zsdkap_zek103_mb52_merged_df = pd.merge(zsdkap_zek103_merged_df, mb52_df, left_on=['delivery_plant', 'mat_number'], right_on=['plant', 'mat_number']).drop(columns=['plant'])

In [39]:
zsdkap_zek103_mb52_merged_df.head(10)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity
0,0301,2044034,2026-07-09,1.0,ZAR M 078/xxx R4R7WL,0.0,NaT,3.0
1,0301,283953,2026-07-10,3.0,ZAR M 114/xxx 4373,0.0,NaT,19.0
2,0301,283953,2026-07-13,3.0,ZAR M 114/xxx 4373,0.0,NaT,19.0
3,0301,283953,2026-07-14,2.0,ZAR M 114/xxx 4373,0.0,NaT,19.0
4,0301,283953,2026-07-16,1.0,ZAR M 114/xxx 4373,0.0,NaT,19.0
5,0301,283953,2026-07-17,0.0,ZAR M 114/xxx 4373,20.0,2026-07-15,19.0
6,0301,283954,2026-07-10,1.0,ZAR M 054/xxx 4373,0.0,NaT,3.0
7,0301,283954,2026-07-13,1.0,ZAR M 054/xxx 4373,0.0,NaT,3.0
8,0301,283954,2026-07-15,0.0,ZAR M 054/xxx 4373,3.0,2026-07-13,3.0
9,0301,283954,2026-07-16,2.0,ZAR M 054/xxx 4373,0.0,NaT,3.0


In [42]:
zsdkap_zek103_mb52_merged_df.to_excel(output_files['df'], index=False)

In [43]:
final_df = zsdkap_zek103_mb52_merged_df.copy()

In [ ]:
# Sort the rows by plant, material number, and date
final_df = final_df.sort_values(by=['delivery_plant', 'mat_number', 'dispatch_date']).reset_index(drop=True)

In [ ]:
# Calculate the running totals (cumulative sums) for orders and POs within each group
final_df['cum_orders'] = final_df.groupby(['delivery_plant', 'mat_number'])['orders_quantity'].cumsum()
final_df['cum_po'] = final_df.groupby(['delivery_plant', 'mat_number'])['po_quantity'].cumsum()

# Compute the final stock left column
final_df['stock_left'] = final_df['stock_quantity'] + final_df['cum_po'] - final_df['cum_orders']

In [45]:
# 6. Drop the temporary cumulative columns
final_df = final_df.drop(columns=['cum_orders', 'cum_po'])

In [52]:
final_df.head(20)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,stock_left
0,0301,2044034,2026-07-09,1.0,ZAR M 078/xxx R4R7WL,0.0,NaT,3.0,2.0
1,0301,283953,2026-07-10,3.0,ZAR M 114/xxx 4373,0.0,NaT,19.0,16.0
2,0301,283953,2026-07-13,3.0,ZAR M 114/xxx 4373,0.0,NaT,19.0,13.0
3,0301,283953,2026-07-14,2.0,ZAR M 114/xxx 4373,0.0,NaT,19.0,11.0
4,0301,283953,2026-07-16,1.0,ZAR M 114/xxx 4373,0.0,NaT,19.0,10.0
5,0301,283953,2026-07-17,0.0,ZAR M 114/xxx 4373,20.0,2026-07-15,19.0,30.0
6,0301,283954,2026-07-10,1.0,ZAR M 054/xxx 4373,0.0,NaT,3.0,2.0
7,0301,283954,2026-07-13,1.0,ZAR M 054/xxx 4373,0.0,NaT,3.0,1.0
8,0301,283954,2026-07-15,0.0,ZAR M 054/xxx 4373,3.0,2026-07-13,3.0,4.0
9,0301,283954,2026-07-16,2.0,ZAR M 054/xxx 4373,0.0,NaT,3.0,2.0
